# Categorizer Fine-tune (Colab GPU) — T027

**HUMAN-IN-THE-LOOP**: Run this notebook on Google Colab with a GPU runtime.
All other Phase 2 tasks (model-server, gate, baseline, zero-shot) are agent-built.
This notebook produces the champion artifact.

## Dataset
Source: `laramee26openBankTransactionData.xlsx` — **UK / GBP-scoped** open banking
transactions. Data cleaning + the locked **v2.0.0 taxonomy (18 categories)** are done
by `training/prepare_dataset.py`, which writes the train/val/test/holdout parquets this
notebook reads. Do NOT re-clean here — just consume the parquets so the split (and the
frozen holdout the gate uses) stays byte-identical to what the baseline trained on.

Notable: input text = `Transaction Description` + a bracketed `Transaction Type` code
(e.g. `LIDL GB NOTTINGHA [DEB]`); `amazon` is its own class (dataset quirk); the thin
classes (fitness/clothes/hotels) are governed by the always_review rule. See the model
card + `docs/DECISIONS.md` (2026-06-16).

## What this notebook does
1. Mounts Google Drive and loads the split from `training/data/` (must exist from `prepare_dataset.py`).
2. Fine-tunes DistilBERT on the training split (full fine-tune, all layers).
3. Applies temperature scaling on the validation set for calibration.
4. Evaluates on holdout — **DO NOT use holdout for any model selection, only final reporting**.
5. Exports: `champion.onnx` (with temperature scalar baked in) + `champion_tokenizer.json`.
6. Prints the SHA-256 to pin in `modelserver/config.py`.

## After this notebook
```bash
# Copy outputs to the repo:
cp champion.onnx training/data/champion.onnx
cp champion_tokenizer.json training/data/champion_tokenizer.json

# Run export:
python training/export_onnx.py --mode champion

# Commit the artifact via LFS (T030) and fill eval_thresholds.yaml (T031).
# The champion must beat the classical baseline (holdout macro-F1 0.8934) by the
# committed margin AND clear the 0.84 floor, or the gate blocks the swap.
```

In [ ]:
# Step 0: GPU check
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else 'No GPU detected — switch to GPU runtime!')

In [ ]:
# Step 1: Install deps (Colab-only; torch + transformers are NEVER in the serving image)
!pip install -q transformers datasets optimum[onnxruntime] onnx onnxruntime pandas pyarrow pyyaml scikit-learn

In [ ]:
# Step 2: Mount Drive and set paths
from google.colab import drive
drive.mount('/content/drive')

# Adjust this to where you cloned/copied the repo
REPO_ROOT = '/content/drive/MyDrive/raseed'
DATA_DIR = f'{REPO_ROOT}/training/data'
MODEL_NAME = 'distilbert-base-uncased'
NUM_EPOCHS = 5
BATCH_SIZE = 32
SEED = 42

In [ ]:
# Step 3: Load taxonomy and splits
import yaml
import pandas as pd

with open(f'{REPO_ROOT}/training/taxonomy.yaml') as f:
    taxonomy = yaml.safe_load(f)
CATEGORIES = taxonomy['categories']
NUM_LABELS = len(CATEGORIES)
label2id = {c: i for i, c in enumerate(CATEGORIES)}
id2label = {i: c for i, c in enumerate(CATEGORIES)}
print(f'Categories ({NUM_LABELS}): {CATEGORIES}')

train_df = pd.read_parquet(f'{DATA_DIR}/train.parquet')
val_df   = pd.read_parquet(f'{DATA_DIR}/val.parquet')
test_df  = pd.read_parquet(f'{DATA_DIR}/test.parquet')
print(f'Train: {len(train_df)}  Val: {len(val_df)}  Test: {len(test_df)}')

In [ ]:
# Step 4: Build HuggingFace datasets
from datasets import Dataset
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(batch['description'], truncation=True, max_length=128, padding='max_length')

def to_hf_dataset(df):
    ds = Dataset.from_pandas(df[['description', 'category']].copy())
    ds = ds.map(lambda x: {'labels': label2id[x['category']]}, remove_columns=[])
    return ds.map(tokenize, batched=True)

train_ds = to_hf_dataset(train_df)
val_ds   = to_hf_dataset(val_df)
print('Datasets ready.')

In [ ]:
# Step 5: Fine-tune DistilBERT (full fine-tune, all layers)
import torch
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import f1_score
import numpy as np

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=NUM_LABELS, id2label=id2label, label2id=label2id
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    macro = f1_score(labels, preds, average='macro', zero_division=0)
    return {'macro_f1': macro}

args = TrainingArguments(
    output_dir='/tmp/distilbert-categorizer',
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=64,
    learning_rate=2e-5,
    weight_decay=0.01,
    evaluation_strategy='epoch',
    save_strategy='best',
    load_best_model_at_end=True,
    metric_for_best_model='macro_f1',
    greater_is_better=True,
    fp16=torch.cuda.is_available(),
    seed=SEED,
    report_to='none',
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
)

trainer.train()
print('Fine-tuning complete.')

In [ ]:
# Step 6: Temperature scaling calibration on validation set
# Learns a single scalar T such that logits/T → better-calibrated probabilities.
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

class TemperatureScaler(nn.Module):
    def __init__(self):
        super().__init__()
        self.temperature = nn.Parameter(torch.ones(1) * 1.5)
    def forward(self, logits):
        return logits / self.temperature

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.eval()
model.to(device)

# Collect val logits
all_logits, all_labels = [], []
val_ds_pt = val_ds.with_format('torch')
loader = DataLoader(val_ds_pt, batch_size=64)
with torch.no_grad():
    for batch in loader:
        input_ids = batch['input_ids'].to(device)
        attn = batch['attention_mask'].to(device)
        out = model(input_ids=input_ids, attention_mask=attn)
        all_logits.append(out.logits.cpu())
        all_labels.append(batch['labels'].cpu())

all_logits = torch.cat(all_logits)
all_labels = torch.cat(all_labels)

# Fit temperature
scaler = TemperatureScaler()
optimizer_t = torch.optim.LBFGS([scaler.temperature], lr=0.01, max_iter=50)
nll = nn.CrossEntropyLoss()

def _eval():
    optimizer_t.zero_grad()
    loss = nll(scaler(all_logits), all_labels)
    loss.backward()
    return loss

optimizer_t.step(_eval)
T = float(scaler.temperature.item())
print(f'Learned temperature: {T:.4f}')

In [ ]:
# Step 7: Evaluate on holdout (final reporting only — holdout is FROZEN)
holdout_df = pd.read_parquet(f'{DATA_DIR}/holdout.parquet')
holdout_ds = to_hf_dataset(holdout_df).with_format('torch')
holdout_loader = DataLoader(holdout_ds, batch_size=64)

holdout_logits, holdout_labels = [], []
with torch.no_grad():
    for batch in holdout_loader:
        input_ids = batch['input_ids'].to(device)
        attn = batch['attention_mask'].to(device)
        out = model(input_ids=input_ids, attention_mask=attn)
        holdout_logits.append(out.logits.cpu())
        holdout_labels.append(batch['labels'].cpu())

holdout_logits = torch.cat(holdout_logits)
calibrated_logits = scaler(holdout_logits)
holdout_preds = calibrated_logits.argmax(dim=-1).numpy()
holdout_true = torch.cat(holdout_labels).numpy()

macro_f1 = f1_score(holdout_true, holdout_preds, average='macro', zero_division=0)
per_class = f1_score(holdout_true, holdout_preds, average=None, zero_division=0)
print(f'Holdout macro-F1 (calibrated): {macro_f1:.4f}')
for cat, sc in zip(CATEGORIES, per_class):
    print(f'  {cat}: {sc:.4f}')

In [ ]:
# Step 8: Export champion to ONNX with temperature scalar baked in
import torch.onnx
from pathlib import Path

OUT_DIR = Path('/tmp/champion_export')
OUT_DIR.mkdir(exist_ok=True)

# Wrap model with temperature scaler
class CalibratedModel(nn.Module):
    def __init__(self, base, temperature):
        super().__init__()
        self.base = base
        self.temperature = nn.Parameter(torch.tensor([temperature]))
    def forward(self, input_ids, attention_mask):
        out = self.base(input_ids=input_ids, attention_mask=attention_mask)
        return out.logits / self.temperature

calibrated_model = CalibratedModel(model, T).to('cpu')
calibrated_model.eval()

# Dummy inputs for tracing
dummy_input_ids = torch.zeros(1, 128, dtype=torch.long)
dummy_attn = torch.ones(1, 128, dtype=torch.long)

torch.onnx.export(
    calibrated_model,
    (dummy_input_ids, dummy_attn),
    str(OUT_DIR / 'champion.onnx'),
    input_names=['input_ids', 'attention_mask'],
    output_names=['logits'],
    dynamic_axes={
        'input_ids': {0: 'batch_size'},
        'attention_mask': {0: 'batch_size'},
        'logits': {0: 'batch_size'},
    },
    opset_version=14,
)
print(f'ONNX exported → {OUT_DIR}/champion.onnx')

# Save tokenizer
tokenizer.save_pretrained(str(OUT_DIR))
# The tokenizers-lib compatible tokenizer.json is in OUT_DIR/tokenizer.json
print(f'Tokenizer saved → {OUT_DIR}/tokenizer.json')

In [ ]:
# Step 9: Compute and print SHA-256 (pin this in config.py as MODELSERVER_EXPECTED_SHA256)
import hashlib

def sha256_of(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(65536), b''):
            h.update(chunk)
    return h.hexdigest()

sha = sha256_of(OUT_DIR / 'champion.onnx')
print('\n=== Champion artifact SHA-256 ===' )
print(f'{sha}')
print('===================================')
print(f'\nHoldout macro-F1: {macro_f1:.4f}')
print(f'Temperature:      {T:.4f}')
print('\nNext steps:')
print(f'  1. Copy {OUT_DIR}/champion.onnx → training/data/champion.onnx')
print(f'  2. Copy {OUT_DIR}/tokenizer.json → training/data/champion_tokenizer.json')
print('  3. python training/export_onnx.py --mode champion')
print('  4. Pin the SHA above in modelserver/config.py (T030)')
print('  5. Fill eval_thresholds.yaml with the numbers above (T031)')